# 04 – Step Response Metrics

**Manufacturing context:** After a step command to a positioning axis, quality is measured by:
- **Rise time** – how fast it arrives (10% to 90%)
- **Settling time** – how quickly oscillations die (within 2% band)
- **Percent overshoot** – how much it exceeds the target
- **Steady-state error** – final accuracy

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ---- Editable parameters ----
m = 1.0      # Mass [kg]
c = 1.0      # Damping [N*s/m] - gives visible overshoot
k = 16.0     # Stiffness [N/m]
F_step = 1.0 # Step force [N]
ref = F_step / k   # Expected steady-state position
t_end = 8.0
dt = 0.001
# -----------------------------

In [ ]:
def rise_time(t, y, final_value):
    """Time from 10% to 90% of final value."""
    y_norm = (y - y[0]) / (final_value - y[0])
    idx_10 = np.where(y_norm >= 0.10)[0]
    idx_90 = np.where(y_norm >= 0.90)[0]
    if len(idx_10) == 0 or len(idx_90) == 0:
        return None
    return float(t[idx_90[0]] - t[idx_10[0]])

def settling_time(t, y, final_value, tol=0.02):
    """Time after which response stays within tol of final value."""
    within = np.abs(y - final_value) <= tol * abs(final_value)
    for i in range(len(within) - 1, -1, -1):
        if not within[i]:
            return float(t[i + 1]) if i < len(within) - 1 else None
    return float(t[0])

def percent_overshoot(y, final_value):
    """Peak overshoot as a percentage."""
    peak = np.max(y)
    if peak <= final_value:
        return 0.0
    return float((peak - final_value) / abs(final_value) * 100.0)

def steady_state_error(y, ref):
    """Absolute error between final output and reference."""
    return abs(float(np.asarray(ref).flat[-1]) - float(y[-1]))

In [ ]:
# Simulate second-order step response
t = np.arange(0, t_end, dt)
y = np.zeros_like(t)
v = np.zeros_like(t)

for i in range(1, len(t)):
    a = (F_step - c * v[i-1] - k * y[i-1]) / m
    v[i] = v[i-1] + a * dt
    y[i] = y[i-1] + v[i] * dt

# Compute metrics
tr  = rise_time(t, y, ref)
ts  = settling_time(t, y, ref)
po  = percent_overshoot(y, ref)
sse = steady_state_error(y, ref)

print("-- Step Response Metrics --")
print(f"  Rise time (10-90%)   : {tr:.4f} s" if tr else "  Rise time: N/A")
print(f"  Settling time (2%)   : {ts:.4f} s" if ts else "  Settling time: N/A")
print(f"  Percent overshoot    : {po:.1f} %" if po is not None else "  Overshoot: N/A")
print(f"  Steady-state error   : {sse:.6f}")

In [ ]:
plt.figure(figsize=(8, 3.5))
plt.plot(t, y, label="Output")
plt.axhline(ref, color="k", linestyle="--", linewidth=0.8, label="Reference")
if tr is not None:
    plt.axvspan(0, tr, alpha=0.05, color="green", label=f"Rise time = {tr:.3f}s")
if ts is not None:
    plt.axvline(ts, color="red", linestyle=":", linewidth=0.8, label=f"Settling = {ts:.3f}s")
plt.xlabel("Time [s]")
plt.ylabel("Position [m]")
plt.title("Step Response with Performance Metrics")
plt.legend(fontsize=8)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Student Exercise

1. Increase `c` to 4.0. How do the metrics change?
2. Which metric matters most for a high-speed packaging machine? For a precision grinding machine?
3. Can you find a `c` value that keeps overshoot below 5% while keeping rise time under 0.5s?